In [42]:
import pandas as pd
import numpy as np

from scipy.stats import randint

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import make_scorer, precision_score
from sklearn.feature_selection import SelectFromModel

# Preprocessing Pipeline Plan

- Limit observations to only the first encounter
- Drop additional prescribed medication due to limited clinical relevance
- Conduct feature decomposition on insulin and metformin to include a binary flag for being prescribed the medication and an ordinal encoding for the change
- Create train, validation and test splits
- Preprocess features
    - Convert data types (admission source, admission type)
    - Scale numeric features
    - Encode categoricals (OHE and ordinal)
    - Encode target (multi-class, binary)

## Experimenting with Feature Decomposition

Decomposing the insulin and metformin columns into two each: one binary flag to indicate if someone is on the drug at all, another ordinal encoding encoding if the patient is on the drug to indicate if there was a change in dosing.

In [43]:
features = pd.read_parquet("data/diabetes_features.parquet")
target = pd.read_parquet("data/diabetes_target.parquet")

### Limiting the Data to the First Encounter Only

In [44]:
# Creating a dataframe for selecting first encounters
splitting_df = features.copy()
splitting_df['target'] = (target['readmitted'] == "<30").astype(int)

# Keeping only the first encounter per patient_nbr
split_df = splitting_df.drop_duplicates('patient_nbr', keep='first')

### Preparing Features

Binary and Ordinal Splits for Insulin and Metformin

In [45]:
# Creating a feature to indicate if a patient is not on insulin or metformin
split_df['missing_insulin'] = (split_df.insulin == "No").astype(int)
split_df['missing_metformin'] = (split_df.metformin == "No").astype(int)

# Creating a dictionary to encode changes in insulin and metformin
encode_dict = {'No': 0,
               "Steady": 0,
               "Up": 1,
               "Down": -1}

# Applying the encoding to engineer new features
split_df['metformin_change'] = split_df['metformin'].apply(lambda x: encode_dict[x])
split_df['insulin_change'] = split_df['insulin'].apply(lambda x: encode_dict[x])

Binary missingness encoding for weight

In [46]:
split_df['missing_weight'] = (split_df['weight'] == "?").astype("int")

Binary indicator for missingness to one-hot encode:
- medical_specialty
- payer_code

In [47]:
# Utility function for encoding missing values in a column for one-hot encoding
def missing_cleaner(x, missing_code, encoding=""):
    if x == missing_code:
        return encoding
    else:
        return x

split_df['payer_code_cleaned'] = split_df['payer_code'].apply(missing_cleaner, args=("?", "missing_payer"))
split_df['medical_specialty_cleaned'] = split_df['medical_specialty'].apply(missing_cleaner, args=("?", "missing_medical_specialty"))

Binary indicator and ordinal encoding for A1C and max_glu_serum

In [48]:
split_df['missing_a1c'] = (split_df['A1Cresult'] == "None").astype("int")
split_df['missing_max_glu_serum'] = (split_df['max_glu_serum'] == "None").astype("int")

#### Creating Train and Test Splits

In [49]:
# List of features to drop
drop_features = ['patient_nbr', 'encounter_id', 'target',
                 'repaglinide', 'nateglinide', 'chlorpropamide',
                 'glimepiride', 'acetohexamide', 'glipizide',
                 'glyburide', 'tolbutamide',
                 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
                 'tolazamide', 'examide', 'citoglipton', 'insulin',
                 'glyburide.metformin', 'glipizide.metformin',
                 'glimepiride.pioglitazone', 'metformin.rosiglitazone',
                 'metformin.pioglitazone', 'metformin', 'insulin',
                 'A1Cresult', 'weight', 'diag_1', 'diag_2', 'diag_3',
                 'max_glu_serum', 'change']

# Building X & y dataframes
y = split_df['target']
X = split_df.drop(columns=drop_features)

# Creating train and test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

## Setting up the Pipeline

Features for scaling:
- time_in_hospital
- num_lab_procedures
- num_procedures
- num_medications
- number_outpatient
- number_inpatient
- number_diagnoses

Features for OHE:
- race
- gender
- admission_type_id
- dicharge_disposition_id
- admission_source_id
- payer_code
- medical_specialty

Features for ordinal encoding:
- Age
- Metformin (already encoded)
- Insulin (already encoded)

#### OHE Columns

In [50]:
OHEFEATURES = ['race', 'gender', 'admission_type_id',
               'discharge_disposition_id', 'admission_source_id',
               'payer_code', 'medical_specialty']

# Pipeline for OHE
ohe_pipeline = Pipeline([('handle_outliers', SimpleImputer(strategy='most_frequent')),
                         ('encoding', OneHotEncoder(handle_unknown='ignore',
                                                    min_frequency=0.01,
                                                    sparse_output=False
                                                    ))])

#### Columns for Scaling

In [51]:
SCALING_FEATURES = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                    'num_medications', 'number_outpatient', 'number_inpatient', 'number_diagnoses']

# Pipeline for Scaling
scaling_pipeline = Pipeline([('scaling', MinMaxScaler())])

#### Columns for Ordinal Encoding

In [52]:
# Setting up a list of columns for ordinal encoding
ORDINAL_COLS = ['age']

# Creating a list of categories for the age column
age_categories = sorted(split_df['age'].unique())

# Setting up a pipeline for ordinal encoding
ordinal_pipeline = Pipeline([('ordinal_encoding', OrdinalEncoder(categories=[age_categories]))])

#### Setting up the Column Transformer

In [53]:
ct = ColumnTransformer([('OHE', ohe_pipeline, OHEFEATURES),
                        ('scaling', scaling_pipeline, SCALING_FEATURES),
                        ('ordinal_encoding', ordinal_pipeline, ORDINAL_COLS)])

# Setting the output to return a pandas dataset
ct.set_output(transform='pandas')

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('OHE', ...), ('scaling', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name`

## Fitting the Model

In [54]:
# Setting up a dictionary of models
models = {'random_forest': RandomForestClassifier(class_weight='balanced'),
          'log_reg': LogisticRegression()}

# Creating the parameter grid
param_grid = {
    'random_forest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 5, 10],
        'model__min_samples_split': [2, 5]
    },
    'log_reg': {
        'model__C': np.logspace(-3, 2, 6)
    }
}

# Setting up a scoring dictionary
scoring_dict = {
    'average_precision': 'average_precision',
    'recall': 'recall',
    'roc_auc': 'roc_auc',
    'precision': make_scorer(precision_score, zero_division=0)
    }

In [55]:
if False:
    # Setting up a StratifiedKFold for cross-validation
    cv = StratifiedKFold(n_splits=5,
                        random_state=42,
                        shuffle=True)

    # Creating a dictionary for CV results
    cv_results = {}


    # Setting up the loop to evaluate the different models across the parameter grids
    for model_name, model in models.items():

        # Pipeline for model training
        model_training_pipeline = Pipeline([('preprocessing', ct),
                                            ('model', model)])

        # Setting up the grid for CV
        grid = GridSearchCV(model_training_pipeline,
                            param_grid=param_grid[model_name],
                            scoring=scoring_dict,
                            refit='average_precision',
                            cv=cv)

        # Storing the results
        cv_results[model_name] = grid.fit(X_train, y_train)

In [56]:
if False:    
    results_df = pd.concat(
        {name: pd.DataFrame(grid.cv_results_) for name, grid in cv_results.items()},
        names=['model']
    ).reset_index(level='model')

    # The columns you actually care about:
    cols = ['model', 'params', 'mean_test_average_precision',
            'mean_test_recall', 'mean_test_roc_auc', 'rank_test_average_precision']
    results_df[cols].sort_values('mean_test_average_precision', ascending=False)

#### CV Findings:

While Logistic regression produces similar results for average precision, its recall is extraoridarily low. This is not acceptable since recall measures how well the model detects patients that are likely to readmit.

Performance of Random Forest on average precision is best with deeper trees and a smaller number of samples for a split. However, these deep trees have a lower recall than shallow trees, suggesting the model is overfitting. Improved recall with a lower number of samples suggests the model may be learning edge cases for a smaller number of patients, or is subject overfitting.

## Model Refinement

Hyperparameter tuning and feature selection

For next time: sort out evaluation metrics, recall is still terrible

In [57]:
# Setting up a new pipeline for hyperparameter tuning
hyperparameter_pipe = Pipeline([('preprocessing', ct),
                                ('model', RandomForestClassifier(random_state=42))])

# Creating a randomized parameter grid
hyper_grid = {'model__max_depth': randint(2, 15),
              'model__min_samples_split': randint(2, 10),
              'model__n_estimators': randint(50, 300),
              'model__max_features': ['sqrt', 'log2', 0.3, 0.5]}

# Setting up a random grid search object
random_cv = RandomizedSearchCV(estimator=hyperparameter_pipe,
                               cv=5,
                               param_distributions=hyper_grid,
                               n_iter=50,
                               scoring=scoring_dict,
                               random_state=42,
                               refit='average_precision')

# Fitting the random grid
random_cv.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': <scipy.stats....t 0x11d4217b0>, 'model__max_features': ['sqrt', 'log2', ...], 'model__min_samples_split': <scipy.stats....t 0x11dcd4b90>, 'model__n_estimators': <scipy.stats....t 0x11dd7e530>}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","{'average_precision': 'average_precision', 'precision': make_scorer(p...ro_division=0), 'recall': 'recall', 'roc_auc': 'roc_auc'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'average_precision'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In alloth

In [58]:
# Dataframe for CV results
pd.DataFrame(random_cv.cv_results_).sort_values('mean_test_average_precision', ascending=False).head(10)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__max_depth,param_model__max_features,param_model__min_samples_split,param_model__n_estimators,params,split0_test_average_precision,...,std_test_roc_auc,rank_test_roc_auc,split0_test_precision,split1_test_precision,split2_test_precision,split3_test_precision,split4_test_precision,mean_test_precision,std_test_precision,rank_test_precision
27,2.713027,0.038804,0.220592,0.005100,11,sqrt,5,237,"{'model__max_depth': 11, 'model__max_features'...",0.175956,...,0.007751,5,0.0,0.0,1.000000,0.0,0.0,0.200000,0.400000,21
6,1.898511,0.026497,0.205338,0.003773,13,log2,7,179,"{'model__max_depth': 13, 'model__max_features'...",0.176369,...,0.005949,2,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,23
44,2.254254,0.016746,0.227685,0.003061,11,log2,5,254,"{'model__max_depth': 11, 'model__max_features'...",0.177087,...,0.007356,4,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,23
7,3.268804,0.064970,0.283340,0.008103,13,sqrt,2,253,"{'model__max_depth': 13, 'model__max_features'...",0.177095,...,0.005082,1,1.0,0.0,0.666667,0.5,0.0,0.433333,0.388730,17
29,1.716832,0.017452,0.163050,0.003135,14,sqrt,2,120,"{'model__max_depth': 14, 'model__max_features'...",0.178475,...,0.005564,15,1.0,0.0,0.600000,0.8,0.5,0.580000,0.337046,8
28,2.701000,0.012023,0.238127,0.003630,13,sqrt,2,206,"{'model__max_depth': 13, 'model__max_features'...",0.177370,...,0.005085,3,1.0,0.0,0.666667,0.5,0.0,0.433333,0.388730,17
9,1.186037,0.008273,0.112756,0.016482,10,sqrt,4,108,"{'model__max_depth': 10, 'model__max_features'...",0.173797,...,0.008108,12,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,23
8,2.612460,0.027363,0.257205,0.005641,11,log2,6,285,"{'model__max_depth': 11, 'model__max_features'...",0.175759,...,0.007258,7,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,23
43,0.849427,0.003672,0.083844,0.001521,12,sqrt,8,64,"{'model__max_depth': 12, 'model__max_features'...",0.174381,...,0.006769,8,0.0,0.0,1.000000,0.0,1.0,0.400000,0.489898,19
36,1.067409,0.007003,0.124676,0.002443,13,log2,2,97,"{'model__max_depth': 13, 'model__max_features'...",0.176095,...,0.006634,28,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,23
